
# Lun Rathana

# Exercise 1: Build the `CSVReader` Class

The `CSVReader` class encapsulates loading a CSV file and previewing its first rows.

In [12]:
from pathlib import Path
from abc import ABC, abstractmethod
import pandas as pd


class CSVReader:
    def __init__(self, file_path: str | Path):
        self.file_path = Path(file_path)
        if not self.file_path.is_file():
            raise FileNotFoundError(f"CSV file not found: {self.file_path}")
        self.data = pd.read_csv(self.file_path)

    def preview(self, n: int = 5) -> pd.DataFrame:
        if n < 0:
            raise ValueError("n must be greater than or equal to 0")
        preview_data = self.data.head(n)
        print(preview_data)
        return preview_data


reader = CSVReader('sample_data (2).csv')
print(f"Loaded file: {reader.file_path}")
print(f"Rows: {len(reader.data)}, Columns: {len(reader.data.columns)}")
reader.preview(5)

Loaded file: sample_data (2).csv
Rows: 7, Columns: 7
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
3   4    David   NaN        NaN       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0


# Exercise 2: Strategy Pattern for Missing Values

A `DataCleaner` switches between interchangeable missing-value strategies without changing its main code.

In [13]:
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return a cleaned copy of the DataFrame."""
        raise NotImplementedError


class DropMissing(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.dropna().reset_index(drop=True)


class FillMean(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        numeric_columns = result.select_dtypes(include='number').columns
        result[numeric_columns] = result[numeric_columns].fillna(
            result[numeric_columns].mean()
        )
        return result


class FillMode(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        for column in result.columns:
            modes = result[column].mode(dropna=True)
            if not modes.empty:
                result[column] = result[column].fillna(modes.iloc[0])
        return result


class DataCleaner:
    def __init__(self, strategy: MissingValueStrategy):
        self.strategy = strategy

    def set_strategy(self, strategy: MissingValueStrategy) -> None:
        self.strategy = strategy

    def clean(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.strategy.handle(df)

In [14]:
cleaner = DataCleaner(DropMissing())
drop_result = cleaner.clean(reader.data)
print('DropMissing result:')
display(drop_result)

cleaner.set_strategy(FillMean())
mean_result = cleaner.clean(reader.data)
print('FillMean result:')
display(mean_result)

cleaner.set_strategy(FillMode())
mode_result = cleaner.clean(reader.data)
print('FillMode result:')
display(mode_result)

assert drop_result.isna().sum().sum() == 0
assert mean_result[['age', 'height_cm', 'weight_kg', 'score']].isna().sum().sum() == 0
assert mode_result.isna().sum().sum() == 0
print('All missing-value strategies completed successfully.')

DropMissing result:


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,5,Eva,27.0,160.0,55.0,New York,88.0


FillMean result:


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0


FillMode result:


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,27.0,172.0,55.0,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,75.0
3,4,David,27.0,160.0,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,75.0
6,7,Grace,27.0,162.0,60.0,Chicago,92.0


All missing-value strategies completed successfully.


# Exercise 3: Decorators for Logging and Timing

Decorators add reusable logging and timing behavior without changing the core methods.

In [15]:
from functools import wraps
from time import perf_counter

def log_action(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Starting {func.__name__}...")
        result = func(*args, **kwargs)
        print(f"Finished {func.__name__}.")
        return result
    return wrapper

def log_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {perf_counter() - start:.6f} seconds.")
        return result
    return wrapper

CSVReader.preview = log_action(log_time(CSVReader.preview))
DataCleaner.clean = log_action(log_time(DataCleaner.clean))

In [16]:
decorated_reader = CSVReader('sample_data (2).csv')
print('Decorated CSVReader.preview:')
decorated_reader.preview(3)

decorated_cleaner = DataCleaner(FillMean())
print('\nDecorated DataCleaner.clean:')
cleaned_data = decorated_cleaner.clean(decorated_reader.data)
display(cleaned_data)

assert decorated_reader.preview.__name__ == 'preview'
assert decorated_cleaner.clean.__name__ == 'clean'
assert cleaned_data.isna().sum().sum() == 0
print('Logging and timing decorators completed successfully.')

Decorated CSVReader.preview:
Starting preview...
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob   NaN      172.0        NaN  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago    NaN
preview took 0.000858 seconds.
Finished preview.

Decorated DataCleaner.clean:
Starting clean...
clean took 0.000930 seconds.
Finished clean.


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0


Logging and timing decorators completed successfully.


# Exercise 4: Factory Pattern for Data Transformations

The factory creates transformation objects from names, while every transformation follows the same `DataTransform` interface.

In [17]:
import pandas as pd
from abc import ABC, abstractmethod

In [18]:
class DataTransform(ABC):
    @abstractmethod
    # Purpose: Forces subclasses to implement a method. Prevents instantiation of the base class.
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """Return a transformed copy of the input DataFrame."""
        raise NotImplementedError


class NormalizeColumns(DataTransform):
    """Normalize numeric columns to the range [0, 1]."""

    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        for column in result.select_dtypes(include='number').columns:
            minimum = result[column].min()
            maximum = result[column].max()
            if pd.notna(minimum) and pd.notna(maximum) and maximum != minimum:
                result[column] = (result[column] - minimum) / (maximum - minimum)
            elif pd.notna(minimum):
                result[column] = 0.0
        return result


class RemoveDuplicates(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        return df.drop_duplicates().reset_index(drop=True)


class StandardizeText(DataTransform):
    # text column: applies strip().lower() to string values
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        result = df.copy()
        for column in result.select_dtypes(include=['object', 'string']).columns:
            result[column] = result[column].map(
                lambda value: value.strip().lower() if isinstance(value, str) else value
            )
        return result


class TransformFactory:
    #One registry maps string names to classes, letting you create transformations dynamically without hardcoding imports or if/else chains.
    _TRANSFORMS = {
        'normalize': NormalizeColumns,
        'remove_duplicates': RemoveDuplicates,
        'duplicates': RemoveDuplicates,
        'standardize_text': StandardizeText,
        'text': StandardizeText,
    }

    @classmethod
    # Purpose: Factory method to create instances of DataTransform subclasses based on a string name.
    def create(cls, transform_name: str) -> DataTransform:
        key = transform_name.strip().lower()
        try:
            return cls._TRANSFORMS[key]()
        # Handle the case where the provided transform name is not in the registry.
        except KeyError as error:
            supported = ', '.join(sorted(cls._TRANSFORMS))
            raise ValueError(
            # Provide a helpful error message with the list of supported transformation names.
                f"Unknown transformation '{transform_name}'. Supported names: {supported}"
            ) from error

In [19]:
# Use the fully cleaned DataFrame produced by Exercise 2 (FillMean strategy).
cleaned_input = cleaned_data.copy()
assert cleaned_input.isna().sum().sum() == 0

normalized_df = TransformFactory.create('normalize').apply(cleaned_input)
deduplicated_df = TransformFactory.create('remove_duplicates').apply(cleaned_input)
standardized_df = TransformFactory.create('text').apply(cleaned_input)

print('Normalized numeric columns:')
display(normalized_df)

print('Duplicate rows removed:')
display(deduplicated_df)

print('Text columns standardized:')
display(standardized_df)

# Transformations can also be composed into a pipeline.
pipeline = ['text', 'normalize', 'remove_duplicates']
transformed_df = cleaned_input.copy()
for transform_name in pipeline:
    transformed_df = TransformFactory.create(transform_name).apply(transformed_df)

print('Composed pipeline result:')
display(transformed_df)

Normalized numeric columns:


,id,name,age,height_cm,weight_kg,city,score
0,0.000000,Alice,0.25000,0.333333,0.433333,New York,0.588235
1,0.166667,Bob,0.40625,0.800000,0.500000,Los Angeles,0.882353
2,0.333333,Charlie,1.00000,0.533333,0.566667,Chicago,0.647059
3,0.500000,David,0.40625,0.466667,0.833333,Houston,0.000000
4,0.666667,Eva,0.00000,0.000000,0.000000,New York,0.764706
5,0.833333,Frank,0.37500,1.000000,1.000000,Los Angeles,0.647059
6,1.000000,Grace,0.40625,0.133333,0.166667,Chicago,1.000000


Duplicate rows removed:


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.00,165.0,68.0,New York,85.0
1,2,Bob,30.25,172.0,70.0,Los Angeles,90.0
2,3,Charlie,35.00,168.0,72.0,Chicago,86.0
3,4,David,30.25,167.0,80.0,Houston,75.0
4,5,Eva,27.00,160.0,55.0,New York,88.0
5,6,Frank,30.00,175.0,85.0,Los Angeles,86.0
6,7,Grace,30.25,162.0,60.0,Chicago,92.0


Text columns standardized:


,id,name,age,height_cm,weight_kg,city,score
0,1,alice,29.00,165.0,68.0,new york,85.0
1,2,bob,30.25,172.0,70.0,los angeles,90.0
2,3,charlie,35.00,168.0,72.0,chicago,86.0
3,4,david,30.25,167.0,80.0,houston,75.0
4,5,eva,27.00,160.0,55.0,new york,88.0
5,6,frank,30.00,175.0,85.0,los angeles,86.0
6,7,grace,30.25,162.0,60.0,chicago,92.0


Composed pipeline result:


,id,name,age,height_cm,weight_kg,city,score
0,0.000000,alice,0.25000,0.333333,0.433333,new york,0.588235
1,0.166667,bob,0.40625,0.800000,0.500000,los angeles,0.882353
2,0.333333,charlie,1.00000,0.533333,0.566667,chicago,0.647059
3,0.500000,david,0.40625,0.466667,0.833333,houston,0.000000
4,0.666667,eva,0.00000,0.000000,0.000000,new york,0.764706
5,0.833333,frank,0.37500,1.000000,1.000000,los angeles,0.647059
6,1.000000,grace,0.40625,0.133333,0.166667,chicago,1.000000


# Exercise 5: Full Cleaning Pipeline

The Template Method pattern defines a consistent load, clean, transform, and save workflow.

In [20]:
class DataPipeline(ABC):
    """Template defining the cleaning pipeline workflow."""

    def run(self) -> pd.DataFrame:
        self.load()
        self.clean()
        self.transform()
        self.save()
        return self.data

    @abstractmethod
    def load(self) -> None:
        raise NotImplementedError

    @abstractmethod
    def clean(self) -> None:
        raise NotImplementedError

    @abstractmethod
    def transform(self) -> None:
        raise NotImplementedError

    @abstractmethod
    def save(self) -> None:
        raise NotImplementedError

In [21]:
class CSVDataPipeline(DataPipeline):
    """Process a CSV using the classes and patterns from Exercises 1-4."""

    def __init__(self, input_path: str | Path, output_path: str | Path):
        self.input_path = Path(input_path)
        self.output_path = Path(output_path)
        self.data: pd.DataFrame | None = None

    @log_action
    @log_time
    def load(self) -> None:
        self.data = CSVReader(self.input_path).data.copy()

    @log_action
    @log_time
    def clean(self) -> None:
        if self.data is None:
            raise RuntimeError('load() must run before clean()')
        self.data = DataCleaner(FillMean()).clean(self.data)

    @log_action
    @log_time
    def transform(self) -> None:
        if self.data is None:
            raise RuntimeError('clean() must run before transform()')
        for transform_name in ('text', 'normalize', 'remove_duplicates'):
            self.data = TransformFactory.create(transform_name).apply(self.data)

    @log_action
    @log_time
    def save(self) -> None:
        if self.data is None:
            raise RuntimeError('transform() must run before save()')
        self.output_path.parent.mkdir(parents=True, exist_ok=True)
        self.data.to_csv(self.output_path, index=False)
        print(f'Saved cleaned data to: {self.output_path}')

In [22]:
output_path = Path('cleaned_sample_data.csv')
pipeline = CSVDataPipeline('sample_data (2).csv', output_path)
pipeline_result = pipeline.run()

print('Final pipeline result:')
display(pipeline_result)

assert output_path.is_file()
assert pipeline_result.isna().sum().sum() == 0
assert list(pd.read_csv(output_path).columns) == list(pipeline_result.columns)
print('Template Method cleaning pipeline completed successfully.')

Starting load...
load took 0.001128 seconds.
Finished load.
Starting clean...
Starting clean...
clean took 0.001195 seconds.
Finished clean.
clean took 0.001216 seconds.
Finished clean.
Starting transform...
transform took 0.001118 seconds.
Finished transform.
Starting save...
Saved cleaned data to: cleaned_sample_data.csv
save took 0.000359 seconds.
Finished save.
Final pipeline result:


,id,name,age,height_cm,weight_kg,city,score
0,0.000000,alice,0.25000,0.333333,0.433333,new york,0.588235
1,0.166667,bob,0.40625,0.800000,0.500000,los angeles,0.882353
2,0.333333,charlie,1.00000,0.533333,0.566667,chicago,0.647059
3,0.500000,david,0.40625,0.466667,0.833333,houston,0.000000
4,0.666667,eva,0.00000,0.000000,0.000000,new york,0.764706
5,0.833333,frank,0.37500,1.000000,1.000000,los angeles,0.647059
6,1.000000,grace,0.40625,0.133333,0.166667,chicago,1.000000


Template Method cleaning pipeline completed successfully.
